In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression  
from sklearn.metrics import mean_squared_error, r2_score,accuracy_score, classification_report, confusion_matrix
from autofeat import AutoFeatRegressor 
import featuretools as ft
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer


In [16]:
df = pd.read_csv('Salary_Data.csv')
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [17]:
# Normaliser les noms de diplômes
education_mapping = {
    "high school": "high school",
    "bachelor's": "bachelor's",
    "bachelor's degree": "bachelor's",
    "master's": "master's",
    "master's degree": "master's",
    "phd": "phd",
    "doctorate": "phd",
    None: "unknown"  # Remplacement des NaN par "unknown"
}

df["Education Level"] = df["Education Level"].str.lower().str.strip().map(education_mapping).fillna("unknown")
education_order = ["unknown", "high school", "bachelor's", "master's", "phd"]

education_encoder = OrdinalEncoder(categories=[education_order])
df["Education Level"] = education_encoder.fit_transform(df[["Education Level"]])

job_encoder = LabelEncoder()
df["Job Title"] = job_encoder.fit_transform(df["Job Title"])


gender_encoder = LabelEncoder()
df["Gender"] = gender_encoder.fit_transform(df["Gender"])

In [18]:
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,2.0,177,5.0,90000.0
1,28.0,0,3.0,18,3.0,65000.0
2,45.0,1,4.0,145,15.0,150000.0
3,36.0,0,2.0,116,7.0,60000.0
4,52.0,1,3.0,26,20.0,200000.0


In [19]:
#salaire en k
df['Salary'] = df.Salary/1000

In [23]:
df.isna().sum()

Age                    2
Gender                 0
Education Level        0
Job Title              0
Years of Experience    3
Salary                 5
dtype: int64

In [25]:
num_cols = ["Age", "Years of Experience", "Salary"]
num_imputer = SimpleImputer(strategy="median")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

cat_cols = ["Gender", "Education Level", "Job Title"]
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])


In [26]:
X = df.drop(columns=["Salary"])  
y = df["Salary"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Erreur quadratique moyenne (MSE) : {mse:.2f}")
print(f"Score R² : {r2:.2f}")

Erreur quadratique moyenne (MSE) : 781.63
Score R² : 0.71


In [27]:
#https://medium.com/@boukamchahamdi/autofeat-automating-feature-engineering-with-python-f22ec23265a9
af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LinearRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

mse = mean_squared_error(y_test, y_pred_af)
r2 = r2_score(y_test, y_pred_af)
print("\n Performance APRÈS AutoFeat")
print(f"Erreur quadratique moyenne (MSE) : {mse:.2f}")
print(f"Score R² : {r2:.2f}")

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:


Nombre de nouvelles features créées : 50

 Performance APRÈS AutoFeat
Erreur quadratique moyenne (MSE) : 533.13
Score R² : 0.80


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
